# Milestone 3 Lab, notebook 3 of 6: Quarantine, and the figures before and after

## Where we are

| Notebook | What you do |
|---|---|
| 1. Where we are, and the problems by hand | Recall the question, find the data problems yourself, and correct the answer key |
| 2. Checks in code | Write two quality checks as queries, and prove each one works |
| **3. Quarantine, and the figures before and after** (you are here) | Set the wrong records aside, recalculate, and confirm the totals add up |
| 4. A to-do list and a helper, by hand | Ask a model to explain the problems, then build an agent that organises its own work |
| 5. A harness works the queue | Give every problem to a ready-made agent harness, and score its reports |
| 6. The same question, with real warnings | Ask the analyst the leader's question, now that the warnings come from real checks |

Model requests in this notebook: **0**. A full run of this lab costs about 10 US cents.

## What this notebook is for

The checks found the bad records. Now each one needs a consequence, and **code** applies it. There are three outcomes:

- **Pass.** The check found nothing.
- **Quarantine.** The records are certainly wrong. They are set aside and left out of the figures.
- **Needs review.** The records are doubtful. They stay in the figures, with a warning, until a person has looked.

You will set the wrong records aside, recalculate the profit, and compare the figures before and after. Then you will check that nothing was lost on the way. There is no AI in this notebook either.

In [1]:

import pandas as pd
from lab_helpers import check_key_against_calculated, load_result, show_doc

from retail_profit_stack.quality.checks import (
    CHECKS,
    number_rows,
    run_record_checks,
    run_totals_check,
)
from retail_profit_stack.quality.quarantine import split_records
from retail_profit_stack.store import get_store
from retail_profit_stack.transform.engines import get_engine
from retail_profit_stack.transform.pipeline import load_sources, run_pipeline, run_transformations

## Step 1: The rules

Which outcome does each problem get? The business decided, in the **BRD**.

In [2]:
show_doc("BRD.md", "5. Business rules")

> **From `docs/BRD.md`: 5. Business rules**

| Rule | Decision |
|---|---|
| Three outcomes | Every check ends in one of three outcomes. **Pass:** nothing wrong. **Quarantine:** the wrong records are set aside and left out of the figures. **Needs review:** the records stay in the figures, and the answer carries a warning |
| Who decides the outcome | Code, by a fixed rule for each check. AI never decides or changes an outcome |
| A record that appears twice | Quarantine. The first copy is kept; every extra copy is set aside |
| A product with no cost for the month | Needs review. As before, its profit is shown as unavailable, never as zero |
| A refund larger than the order line it refunds | Needs review. The refund still counts until a person has looked |
| A refund or shipment dated before its order | Needs review. The record still counts |
| Totals that do not add up | After the calculation, the totals in the profit figures must equal the totals in the records that were kept. A difference needs review |
| Quarantined records | Never deleted. They are kept in their own table so anyone can inspect them |
| The figures change | The trail water bottle's October figures go down, because three duplicated order lines are no longer counted. Its contribution profit moves from $5,188.20 to $5,156.94. The Hydration category total, which includes it, moves by the same $31.26. Every other product's figures stay the same |
| What AI does | It investigates each problem, names the likely cause from the rulebook, and writes a report. It cannot change the data, the figures, or an outcome |
| A problem the AI could not finish | It is marked "left for a person". The run carries on with the others |
| Which models | Only models available through OpenRouter. Nothing is installed on the learner's computer |

Notice the reasoning. An extra copy of a record is **certainly** wrong, so it is quarantined. A large refund or an odd date **might** be right, so a person decides. Only certain errors are removed automatically.

## Step 2: Give each bad record its outcome

Run the record checks again, as in notebook 2. Then look up each bad record's outcome from its check.

In [3]:
sources = number_rows(load_sources())
bad_records = run_record_checks(sources)

outcome_of_check = {check.name: check.outcome for check in CHECKS}
bad_records["outcome"] = bad_records["check_name"].map(outcome_of_check)

bad_records.groupby(["check_name", "outcome"]).size().rename("bad records").reset_index()

,check_name,outcome,bad records
0,duplicate_records,quarantine,22
1,event_out_of_order,needs_review,2
2,missing_product_cost,needs_review,1
3,refund_exceeds_line,needs_review,1


22 records to quarantine, all of them extra copies. Four records need review.

## Step 3: Set records aside, by hand, for one table

Do it for the order lines first, so you can see there is no trick. Take the row numbers of the order lines to quarantine.

In [4]:
lines = sources["order_items"].to_pandas()

is_line_to_quarantine = (bad_records["outcome"] == "quarantine") & (bad_records["source_table"] == "order_items")
row_numbers = bad_records[is_line_to_quarantine]["row_number"]

sorted(int(number) for number in row_numbers)

[1247, 1435, 1437, 1440, 1442, 1756, 1931]

Now split the table in two: the rows with those numbers, and the rest.

In [5]:
quarantined_lines = lines[lines["row_number"].isin(row_numbers)]
kept_lines = lines[~lines["row_number"].isin(row_numbers)]

print("All order lines:", len(lines))
print("Kept:           ", len(kept_lines))
print("Quarantined:    ", len(quarantined_lines))
print("Kept + quarantined = all?", len(kept_lines) + len(quarantined_lines) == len(lines))

All order lines: 2549
Kept:            2542
Quarantined:     7
Kept + quarantined = all? True


The last line matters. **Nothing is deleted.** Every row is in one pile or the other.

In [6]:
quarantined_lines

,order_item_id,order_id,product_id,quantity,unit_list_price,discount_amount,row_number
1246,OI-01246,ORD-21207,P001,1,40.0,4.00,1247
1434,OI-01433,ORD-21389,P002,1,35.0,1.75,1435
1436,OI-01434,ORD-21390,P002,1,35.0,1.75,1437
1439,OI-01436,ORD-21392,P002,3,35.0,5.25,1440
1441,OI-01437,ORD-21393,P002,4,35.0,7.00,1442
1755,OI-01750,ORD-21695,P001,1,40.0,4.00,1756
1930,OI-01924,ORD-21860,P001,1,40.0,4.00,1931


## Step 4: The production version, for every table

`split_records` does what you just did, for all seven tables. It returns the kept tables and the quarantined tables.

In [7]:
kept, quarantined = split_records(sources, bad_records.drop(columns=["outcome"]))

pd.DataFrame([
    {"table": name, "all rows": sources[name].num_rows, "kept": kept[name].num_rows, "quarantined": quarantined[name].num_rows}
    for name in sources
])

,table,all rows,kept,quarantined
0,products,12,12,0
1,orders,2450,2450,0
2,order_items,2549,2542,7
3,refunds,50,49,1
4,shipments,2525,2518,7
5,payment_fees,2525,2518,7
6,product_costs,23,23,0


## Step 5: The figures before and after

Run milestone 2's six transformations twice: once on **all** the records, as milestone 2 did, and once on the **kept** records only. The transformations themselves have not changed at all.

In [8]:
all_records = load_sources()

before_table = run_transformations(get_engine("sql"), all_records)["monthly_product_profit"]
after_table = run_transformations(get_engine("sql"), kept)["monthly_product_profit"]

before = before_table.to_pandas()
after = after_table.to_pandas()

Put the two side by side and keep the rows where something changed.

In [9]:
both = before.merge(after, on=["month", "product_id"], suffixes=(" before", " after"))
changed = (both["units_sold before"] != both["units_sold after"]) | (both["refunds before"] != both["refunds after"])

both[changed][["month", "product_id", "units_sold before", "units_sold after", "refunds before", "refunds after", "contribution_profit before", "contribution_profit after"]]

,month,product_id,units_sold before,units_sold after,refunds before,refunds after,contribution_profit before,contribution_profit after
4,2026-09,P005,48,48,912.00,608.00,2592.24,2926.24
12,2026-10,P001,510,507,108.00,108.00,5188.20,5156.94
13,2026-10,P002,249,240,33.25,33.25,2638.14,2541.35


Three product-months changed:

- **P001, October** (trail water bottle): 3 units fewer, and contribution profit of **5156.94**. That is the figure you worked out by hand in notebook 1.
- **P002, October** (insulated flask): 9 units fewer. Nobody had listed these duplicates. Without the check, this figure would have been wrong and no one would have known.
- **P005, September** (two-person tent): the refund that was counted twice is now counted once.

## Step 6: Did we lose anything? The totals check

We removed records, then ran six transformations. How do we know the profit table still adds up to the kept records, with nothing dropped and nothing counted twice?

The fifth check, `totals_do_not_match`, answers that. For every product, month, and term it adds up the kept records and compares the total with the profit table. It returns every total that differs.

In [10]:
run_totals_check(kept, after_table)

,check_name,month,product_id,source_table,row_number,detail


Empty: every total matches. To see the check catch something, give it the **before** table, which was calculated from all the records, copies included.

In [11]:
pd.set_option("display.max_colwidth", 110)

run_totals_check(kept, before_table)[["month", "product_id", "detail"]]

,month,product_id,detail
0,2026-10,P001,"units_sold: the records add up to 507.00, but the profit table says 510.00."
1,2026-10,P002,"units_sold: the records add up to 240.00, but the profit table says 249.00."
2,2026-10,P001,"gross_sales: the records add up to 20280.00, but the profit table says 20400.00."
3,2026-10,P002,"gross_sales: the records add up to 8400.00, but the profit table says 8715.00."
4,2026-10,P002,"discounts: the records add up to 420.00, but the profit table says 435.75."
5,2026-10,P001,"discounts: the records add up to 2028.00, but the profit table says 2040.00."
6,2026-09,P005,"refunds: the records add up to 608.00, but the profit table says 912.00."
7,2026-09,P005,"return_cost: the records add up to 60.00, but the profit table says 90.00."
8,2026-10,P002,"fulfillment_cost: the records add up to 600.00, but the profit table says 622.50."
9,2026-10,P001,"fulfillment_cost: the records add up to 1267.50, but the profit table says 1275.00."


It names each total that is wrong, and by how much. A check like this is called a **reconciliation**. It is the safety net for the whole calculation.

## Step 7: The whole pipeline

In production, one function does all of this in order. Here is the flow from the **LLD**.

In [12]:
show_doc("LLD.md", "2. One pipeline run, step by step")

> **From `docs/LLD.md`: 2. One pipeline run, step by step**

*(Diagram: open `docs/LLD.md` to see it.)*

`run_pipeline(engine_name=None)` does these six steps in order and writes everything to `data/warehouse/`.

In [13]:
run = run_pipeline("sql")

print(f"Ran on the {run.engine} engine in {run.seconds} seconds")
pd.Series(run.check_outcomes, name="outcome").to_frame()

Ran on the sql engine in 0.11 seconds


,outcome
duplicate_records,quarantine
missing_product_cost,needs_review
event_out_of_order,needs_review
refund_exceeds_line,needs_review
totals_do_not_match,pass


Four checks found something. The totals check passed.

The pipeline groups the bad records into **findings**: one per check, product, and month. This table replaces the hand-written list from milestones 1 and 2.

In [14]:
run.findings[["finding_id", "check_name", "month", "product_id", "outcome", "severity", "record_count"]]

,finding_id,check_name,month,product_id,outcome,severity,record_count
0,QF-001,duplicate_records,2026-09,P005,quarantine,low,1
1,QF-002,duplicate_records,2026-10,P001,quarantine,low,9
2,QF-003,duplicate_records,2026-10,P002,quarantine,low,12
3,QF-004,missing_product_cost,2026-10,P007,needs_review,high,0
4,QF-005,event_out_of_order,2026-09,P009,needs_review,low,1
5,QF-006,event_out_of_order,2026-10,P003,needs_review,low,1
6,QF-007,refund_exceeds_line,2026-09,P011,needs_review,medium,1


In [15]:
for description in run.findings["description"]:
    print("-", description)

- Records for Two-person tent in 2026-09 that appeared more than once: 1. The extra copies were set aside and are not in the figures.
- Records for Trail water bottle in 2026-10 that appeared more than once: 9. The extra copies were set aside and are not in the figures.
- Records for Insulated flask in 2026-10 that appeared more than once: 12. The extra copies were set aside and are not in the figures.
- No product cost is recorded for Camp stove in 2026-10, so its gross profit and contribution profit cannot be calculated. Its category total leaves it out.
- Records for Rain jacket in 2026-09 dated before their order: 1. They are still counted, in the month of their own date.
- Records for Hydration pack in 2026-10 dated before their order: 1. They are still counted, in the month of their own date.
- Refunds for Sun hat in 2026-09 that are larger than the order line they refund: 1. They are still counted, so net sales may be understated.


Those sentences are built by code from a fixed template for each check. No model wrote them.

The quarantined records are saved too, so finance can always see what was left out and why.

In [16]:
set_aside = run.finding_records[run.finding_records["quarantined"]]

print(len(set_aside), "records set aside")
set_aside[["finding_id", "source_table", "row_number", "detail", "record"]].head(4)

22 records set aside


,finding_id,source_table,row_number,detail,record
0,QF-001,refunds,15,This row is copy number 2 of the same record.,"{""refund_id"": ""REF-0014"", ""order_item_id"": ""OI-00514"", ""product_id"": ""P005"", ""refund_date"": ""2026-09-20"", ..."
1,QF-002,order_items,1247,This row is copy number 2 of the same record.,"{""order_item_id"": ""OI-01246"", ""order_id"": ""ORD-21207"", ""product_id"": ""P001"", ""quantity"": 1, ""unit_list_pri..."
2,QF-002,order_items,1756,This row is copy number 2 of the same record.,"{""order_item_id"": ""OI-01750"", ""order_id"": ""ORD-21695"", ""product_id"": ""P001"", ""quantity"": 1, ""unit_list_pri..."
3,QF-002,order_items,1931,This row is copy number 2 of the same record.,"{""order_item_id"": ""OI-01924"", ""order_id"": ""ORD-21860"", ""product_id"": ""P001"", ""quantity"": 1, ""unit_list_pri..."


## Step 8: Does it match the answer key?

The answer key is the one you corrected by hand in notebook 1. The figures below are read from the table the pipeline just wrote, through the same tools the analyst uses.

In [17]:
get_store.cache_clear()  # make the tools load the tables the pipeline just wrote
answer_key = load_result("answer_key", required_from="notebook 1")

check_key_against_calculated(answer_key)

,What,Answer key (by hand),Calculated (by the pipeline),Same?
0,#1 product: Trail water bottle,"$5,156.94","$5,156.94",yes
1,#2 product: Headlamp,"$4,250.00","$4,250.00",yes
2,#3 product: Rain jacket,"$3,841.00","$3,841.00",yes
3,Top category: Hydration,"$13,326.80","$13,326.80",yes
4,"Hydration, change since the month before",+$344.80,+$344.80,yes
5,"Apparel, change since the month before (the biggest fall)","-$5,914.87","-$5,914.87",yes
6,"Camp, change since the month before (cannot be compared)",unavailable,unavailable,yes
7,Camp stove (its cost is missing),unavailable,unavailable,yes


Every line should say **yes**, including the first: the trail water bottle at $5,156.94.

## What we now know

1. Every check has a fixed outcome, and code applies it. Only certain errors are quarantined.
2. Quarantine means set aside, not deleted. Kept plus quarantined always equals the source.
3. The figures changed for three products. Two of those errors were on nobody's list.
4. A totals check after the calculation proves nothing was lost or counted twice.

**Next:** notebook 4. There are seven findings now, and someone has to work out what caused each one. That is where AI comes in.